Imports

In [1]:
import sqlite3
import threading
import queue
import time
import uuid
from datetime import datetime

Database

In [2]:
# ==========================
# DATABASE
# ==========================

conn = sqlite3.connect("notification.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS notifications(

    notification_id TEXT PRIMARY KEY,
    username TEXT,
    channel TEXT,
    message TEXT,
    status TEXT,
    created_at TEXT

)
""")

conn.commit()



In [3]:
#save notification to database

def save_notification(notification):

    cursor.execute("""

    INSERT INTO notifications
    VALUES(?,?,?,?,?,?)

    """,(

        notification["id"],
        notification["user"],
        notification["channel"],
        notification["message"],
        notification["status"],
        notification["time"]

    ))

    conn.commit()

In [4]:
#Update Status

def update_status(notification_id,status):

    cursor.execute("""

    UPDATE notifications
    SET status=?
    WHERE notification_id=?

    """,(status,notification_id))

    conn.commit()

In [5]:
#Display Database

def show_notifications():

    cursor.execute("SELECT * FROM notifications")

    rows=cursor.fetchall()

    print("\nNotification History\n")

    for row in rows:

        print(row)

Queue

In [6]:
# ==========================
# NOTIFICATION QUEUE
# ==========================

notification_queue = queue.Queue()

Validator

In [7]:
# ==========================
# VALIDATOR
# ==========================

class Validator:

    def validate(self,notification):

        if notification["user"]=="":

            return False

        if notification["message"]=="":

            return False

        if notification["channel"] not in ["Email","SMS","Push"]:

            return False

        return True

Logger

In [8]:
# ==========================
# LOGGER
# ==========================

class Logger:

    def log(self,message):

        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

Notification Server (Producer)

This is the Notification API in the architecture. It receives requests, validates them, saves them as Pending, and pushes them into the queue.

In [9]:
# ==========================
# NOTIFICATION SERVER
# ==========================

class NotificationServer:

    def __init__(self):

        self.validator = Validator()
        self.logger = Logger()

    def receive_notification(self, user, channel, message):

        notification = {

            "id": str(uuid.uuid4())[:8],
            "user": user,
            "channel": channel,
            "message": message,
            "status": "Pending",
            "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        }

        if not self.validator.validate(notification):

            self.logger.log("Invalid Notification Request")
            return

        save_notification(notification)

        notification_queue.put(notification)

        self.logger.log(
            f"Notification {notification['id']} added to queue."
        )

Email Service

In [10]:
# ==========================
# EMAIL SERVICE
# ==========================

class EmailService:

    def send(self, notification):

        print(f"📧 Sending Email to {notification['user']}")

        time.sleep(2)

        return "Sent"

SMS Service

In [11]:
# ==========================
# SMS SERVICE
# ==========================

class SMSService:

    def send(self, notification):

        print(f"📱 Sending SMS to {notification['user']}")

        time.sleep(2)

        return "Sent"

Push Notification Service

In [12]:
# ==========================
# PUSH SERVICE
# ==========================

class PushService:

    def send(self, notification):

        print(f"🔔 Sending Push Notification to {notification['user']}")

        time.sleep(2)

        return "Sent"

Channel Router

In [13]:
# ==========================
# CHANNEL ROUTER
# ==========================

class ChannelRouter:

    def __init__(self):

        self.email = EmailService()
        self.sms = SMSService()
        self.push = PushService()

    def route(self, notification):

        channel = notification["channel"]

        if channel == "Email":

            return self.email.send(notification)

        elif channel == "SMS":

            return self.sms.send(notification)

        elif channel == "Push":

            return self.push.send(notification)

        return "Failed"

Notification Processor (Consumer)
This is the heart of the system. It continuously monitors the queue, processes notifications, routes them to the correct service, updates the database, and logs the result.

In [14]:
# ==========================
# NOTIFICATION PROCESSOR
# ==========================

class NotificationProcessor:

    def __init__(self):

        self.router = ChannelRouter()
        self.logger = Logger()

    def process_notifications(self):

        while True:

            notification = notification_queue.get()

            self.logger.log(
                f"Processing {notification['id']}"
            )

            status = self.router.route(notification)

            update_status(notification["id"], status)

            self.logger.log(
                f"{notification['id']} -> {status}"
            )

            notification_queue.task_done()

Start Worker Thread

In [15]:
# ==========================
# START WORKER
# ==========================

processor = NotificationProcessor()

worker = threading.Thread(
    target=processor.process_notifications,
    daemon=True
)

worker.start()

Simulation

In [16]:
# ==========================
# SIMULATION
# ==========================

server = NotificationServer()

server.receive_notification(
    "Sakshi",
    "Email",
    "Welcome to the Notification System!"
)

server.receive_notification(
    "Rahul",
    "SMS",
    "Your OTP is 456321"
)

server.receive_notification(
    "Priya",
    "Push",
    "Your order has been delivered."
)

[04:08:28] Notification 8c0109c7 added to queue.[04:08:28] Processing 8c0109c7
📧 Sending Email to Sakshi

[04:08:28] Notification b832ab48 added to queue.
[04:08:28] Notification cafc4560 added to queue.


Wait for Processing

In [17]:
# Wait for all notifications to be processed

notification_queue.join()

[04:08:30] 8c0109c7 -> Sent
[04:08:30] Processing b832ab48
📱 Sending SMS to Rahul
[04:08:32] b832ab48 -> Sent
[04:08:32] Processing cafc4560
🔔 Sending Push Notification to Priya
[04:08:34] cafc4560 -> Sent


Display Notification History

In [18]:
show_notifications()


Notification History

('8c0109c7', 'Sakshi', 'Email', 'Welcome to the Notification System!', 'Sent', '2026-06-18 04:08:28')
('b832ab48', 'Rahul', 'SMS', 'Your OTP is 456321', 'Sent', '2026-06-18 04:08:28')
('cafc4560', 'Priya', 'Push', 'Your order has been delivered.', 'Sent', '2026-06-18 04:08:28')
